In [1]:
import os
import re
import json
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")
print("Pustaka Pipeline & Deployment berhasil dimuat.")

Pustaka Pipeline & Deployment berhasil dimuat.


In [2]:
# Memuat ketiga tabel data riil
path_cafes   = '../dataset/cafes.csv'
path_reviews = '../dataset/reviews.csv'
path_aspects = '../dataset/aspect_sentiments.csv'

df_cafes   = pd.read_csv(path_cafes)
df_reviews = pd.read_csv(path_reviews)
df_aspects = pd.read_csv(path_aspects)

# Imputasi harga cafe jika ada null
if 'price_min_imputed' not in df_cafes.columns:
    df_cafes['price_min_imputed'] = df_cafes['price_min'].fillna(df_cafes['price_min'].median())
    df_cafes['price_max_imputed'] = df_cafes['price_max'].fillna(df_cafes['price_max'].median())

# Agregasi profil teks per cafe untuk keperluan pencarian rekomendasi
cafe_text_agg = df_reviews.groupby('cafe_id')['text'].apply(lambda x: ' '.join(x.astype(str))).reset_index()
df_cafes_full = pd.merge(df_cafes, cafe_text_agg, left_on='id', right_on='cafe_id', how='left')
df_cafes_full['text'] = df_cafes_full['text'].fillna('')

print(f"Total profil cafe siap rekomendasi: {len(df_cafes_full)}")
print(f"Total data ulasan siap inferensi   : {len(df_reviews):,}")

Total profil cafe siap rekomendasi: 275
Total data ulasan siap inferensi   : 7,736


In [3]:
# Menyiapkan data latih ulasan sentimen
df_sent = pd.merge(df_aspects[['review_id', 'sentiment']], df_reviews[['id', 'text']], left_on='review_id', right_on='id')
df_sent = df_sent.drop_duplicates(subset=['id', 'sentiment']).reset_index(drop=True)

X_train_text = df_sent['text']
y_train_sent = df_sent['sentiment']

# Membangun Pipeline Terpadu: TF-IDF Vectorizer + Linear SVM (Balanced)
cafind_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=1500, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ('classifier', SVC(kernel='linear', C=1.5, class_weight='balanced', probability=True, random_state=42))
])

print("Melatih End-to-End Pipeline Cafind...")
t0 = time.time()
cafind_pipeline.fit(X_train_text, y_train_sent)
t_train = time.time() - t0
print(f"Pelatihan Pipeline Selesai dalam {t_train:.2f} detik.")

# Serialisasi pipeline ke disk
os.makedirs('../outputs', exist_ok=True)
pipeline_file = '../outputs/cafind_sentiment_pipeline.joblib'
joblib.dump(cafind_pipeline, pipeline_file)
print(f"Pipeline berhasil diserialisasi dan disimpan ke: {pipeline_file} ({os.path.getsize(pipeline_file):,} bytes)")

Melatih End-to-End Pipeline Cafind...
Pelatihan Pipeline Selesai dalam 24.58 detik.
Pipeline berhasil diserialisasi dan disimpan ke: ../outputs/cafind_sentiment_pipeline.joblib (1,308,936 bytes)


D:\laragon\bin\python\python-3.13\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



In [4]:
# Memuat pipeline tersimpan dan menguji prediksi
loaded_pipeline = joblib.load(pipeline_file)

test_ulasan_baru = [
    "Tempatnya estetik banget, kopi susunya creamy dan pas buat nongkrong sore",
    "Pelayanannya sangat lambat, pesanan saya kelupaan dan harganya terlalu mahal",
    "Cukup standar seperti warkop biasa, tempat parkir agak sempit"
]

hasil_prediksi = loaded_pipeline.predict(test_ulasan_baru)
probabilitas   = loaded_pipeline.predict_proba(test_ulasan_baru)
classes        = loaded_pipeline.classes_

print("=== VERIFIKASI INFERENSI PIPELINE PADA ULASAN BARU ===")
for i, (teks, pred, prob) in enumerate(zip(test_ulasan_baru, hasil_prediksi, probabilitas), 1):
    prob_dict = {classes[j]: f"{prob[j]*100:.1f}%" for j in range(len(classes))}
    print(f"Ulasan #{i} : '{teks}'")
    print(f"Prediksi  : {pred.upper()} (Probabilitas: {prob_dict})")
    print("-" * 65)

=== VERIFIKASI INFERENSI PIPELINE PADA ULASAN BARU ===
Ulasan #1 : 'Tempatnya estetik banget, kopi susunya creamy dan pas buat nongkrong sore'
Prediksi  : POSITIVE (Probabilitas: {'negative': '0.7%', 'neutral': '18.4%', 'positive': '80.9%'})
-----------------------------------------------------------------
Ulasan #2 : 'Pelayanannya sangat lambat, pesanan saya kelupaan dan harganya terlalu mahal'
Prediksi  : NEGATIVE (Probabilitas: {'negative': '32.0%', 'neutral': '39.8%', 'positive': '28.2%'})
-----------------------------------------------------------------
Ulasan #3 : 'Cukup standar seperti warkop biasa, tempat parkir agak sempit'
Prediksi  : NEUTRAL (Probabilitas: {'negative': '4.4%', 'neutral': '70.1%', 'positive': '25.5%'})
-----------------------------------------------------------------


In [5]:
# Vektorisasi profil ulasan seluruh cafe
tfidf_recommender = TfidfVectorizer(max_features=1000, stop_words='english', min_df=2)
cafe_tfidf_matrix = tfidf_recommender.fit_transform(df_cafes_full['text'])

def rekomendasikan_cafe(kebutuhan_pengguna, budget_maksimal=None, top_n=5):
    # 1. Transformasi teks query preferensi pengguna
    q_vec = tfidf_recommender.transform([kebutuhan_pengguna])
    
    # 2. Perhitungan Cosine Similarity antara query dan profil seluruh cafe
    sim_scores = cosine_similarity(q_vec, cafe_tfidf_matrix).flatten()
    
    # Salin DataFrame cafe untuk kalkulasi skor akhir
    df_result = df_cafes_full.copy()
    df_result['similarity_score'] = sim_scores
    
    # 3. Filter budget jika ditentukan pengguna
    if budget_maksimal is not None:
        df_result = df_result[df_result['price_min_imputed'] <= budget_maksimal]
    
    # 4. Pembobotan skor gabungan: 70% Kemiripan Suasana Teks + 30% Reputasi Rating Google Maps
    rating_normalized = (df_result['rating'] - 1.0) / 4.0
    df_result['final_score'] = (0.70 * df_result['similarity_score']) + (0.30 * rating_normalized)
    
    top_cafes = df_result.sort_values(by='final_score', ascending=False).head(top_n)
    
    output = []
    for rank, (_, row) in enumerate(top_cafes.iterrows(), 1):
        output.append({
            'Peringkat': rank,
            'Nama Cafe': row['name'],
            'Rating': f"{row['rating']:.1f} ★ ({int(row['review_count'])} ulasan)",
            'Rentang Harga': f"Rp {row['price_min_imputed']:,.0f} - {row['price_max_imputed']:,.0f}",
            'Skor Kesesuaian': f"{row['final_score']*100:.1f}%",
            'Alamat': str(row['address'])[:55] + '...'
        })
    return pd.DataFrame(output)

print("Modul Mesin Rekomendasi Cafind Berhasil Diinisialisasi.")

Modul Mesin Rekomendasi Cafind Berhasil Diinisialisasi.


In [6]:
# Skenario Persona 1: Mahasiswa mencari tempat nugas dengan wifi dan colokan
query_1 = "tempat tenang nyaman untuk nugas skripsi kerja laptop wifi kencang banyak colokan"
print(f"=== SKENARIO 1: Persona Mahasiswa (Nugas / WFH) ===")
print(f"Query: '{query_1}'")
print(rekomendasikan_cafe(query_1, budget_maksimal=25000, top_n=4).to_string(index=False))

# Skenario Persona 2: Komunitas nongkrong santai murah
query_2 = "warkop kopi murah meriah buat nongkrong santai malam hari rame rame bareng teman"
print(f"\n=== SKENARIO 2: Persona Komunitas Budget (Nongkrong Murah) ===")
print(f"Query: '{query_2}'")
print(rekomendasikan_cafe(query_2, budget_maksimal=15000, top_n=4).to_string(index=False))

# Skenario Persona 3: Penikmat kopi dan makanan estetis keluarga
query_3 = "cafe estetik mewah makanan lezat dessert kopi enak suasana bersih ramah keluarga"
print(f"\n=== SKENARIO 3: Persona Premium / Kuliner Keluarga ===")
print(f"Query: '{query_3}'")
print(rekomendasikan_cafe(query_3, budget_maksimal=None, top_n=4).to_string(index=False))

=== SKENARIO 1: Persona Mahasiswa (Nugas / WFH) ===
Query: 'tempat tenang nyaman untuk nugas skripsi kerja laptop wifi kencang banyak colokan'
 Peringkat                     Nama Cafe              Rating      Rentang Harga Skor Kesesuaian                                                     Alamat
         1            WARKOP NUSANTARA 2   4.9 ★ (71 ulasan)           Rp 1 - 1           48.2% Jl. Terusan Sulfat No.143, Pandanwangi, Kec. Kedungkand...
         2 Kopi Studio 24 Candi Panggung  4.9 ★ (654 ulasan)           Rp 1 - 1           47.3% Jl. Candi Panggung Bar., Tunggulwulung, Kec. Lowokwaru,...
         3                Robusto Coffee 4.7 ★ (1421 ulasan) Rp 25,000 - 25,000           46.8% Jl. Raya Tlogomas No.2, Dinoyo, Kec. Lowokwaru, Kota Ma...
         4                             -   5.0 ★ (17 ulasan) Rp 25,000 - 25,000           45.2% Jl. Tirtagangga No.11 B, Samaan, Kec. Klojen, Kota Mala...

=== SKENARIO 2: Persona Komunitas Budget (Nongkrong Murah) ===
Query: 'warkop kop

In [7]:
# Menyimpan manifest deployment spesifikasi API
deployment_spec = {
    "project": "Cafind Recommendation Engine",
    "version": "1.0.0",
    "pipeline_model": "cafind_sentiment_pipeline.joblib",
    "dataset_source": "dump-cafind-202611172252.sql",
    "total_cafes_indexed": len(df_cafes_full),
    "endpoints": [
        {
            "route": "/api/recommend",
            "method": "POST",
            "description": "Menerima query preferensi suasana/kebutuhan dan mengembalikan daftar rekomendasi cafe.",
            "request_body": {"query": "string", "budget_max": "float", "top_k": "int"}
        },
        {
            "route": "/api/analyze-sentiment",
            "method": "POST",
            "description": "Mengklasifikasikan sentimen teks ulasan ulasan baru (positive, neutral, negative).",
            "request_body": {"text": "string"}
        }
    ]
}

path_spec = '../outputs/deployment_manifest_js15.json'
with open(path_spec, 'w', encoding='utf-8') as f:
    json.dump(deployment_spec, f, indent=2)

print(f"Manifest spesifikasi deployment tersimpan ke: {path_spec}")
print(json.dumps(deployment_spec, indent=2))

Manifest spesifikasi deployment tersimpan ke: ../outputs/deployment_manifest_js15.json
{
  "project": "Cafind Recommendation Engine",
  "version": "1.0.0",
  "pipeline_model": "cafind_sentiment_pipeline.joblib",
  "dataset_source": "dump-cafind-202611172252.sql",
  "total_cafes_indexed": 275,
  "endpoints": [
    {
      "route": "/api/recommend",
      "method": "POST",
      "description": "Menerima query preferensi suasana/kebutuhan dan mengembalikan daftar rekomendasi cafe.",
      "request_body": {
        "query": "string",
        "budget_max": "float",
        "top_k": "int"
      }
    },
    {
      "route": "/api/analyze-sentiment",
      "method": "POST",
      "description": "Mengklasifikasikan sentimen teks ulasan ulasan baru (positive, neutral, negative).",
      "request_body": {
        "text": "string"
      }
    }
  ]
}
